In [ ]:
import collections
import collections.abc

# แพตช์ (Patch) เพื่อให้โค้ดเก่า (PrivMRF) ทำงานได้บน Python 3.10+
collections.Iterable = collections.abc.Iterable
collections.Mapping = collections.abc.Mapping
collections.MutableSet = collections.abc.MutableSet
collections.MutableMapping = collections.abc.MutableMapping

import matplotlib
matplotlib.use('Agg') 

import warnings
warnings.filterwarnings('ignore')

import os
import csv
import json
import random 
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn import svm

# อย่าลืม !pip install diffprivlib ก่อนรัน
# from diffprivlib.models import KMeans as DPKMeans
from custom_dp_kmeans import KMeans as DPKMeans


# ดึงเครื่องมือของ PrivMRF มาใช้งาน
from exp.evaluate import run_syn 
from PrivMRF.domain import Domain
import PrivMRF.attribute_hierarchy as ah

# ==========================================
# 1. 🎯 กระบวนการเตรียมข้อมูล (N คอลัมน์) ก่อนเข้า PrivMRF
# ==========================================
def prepare_nd_discrete_data(raw_continuous_np, epsilon, num_bins):
    # 1. ทำ Multivariate K-Means หาจุดศูนย์กลางในอวกาศ N มิติ
    min_bounds = np.min(raw_continuous_np, axis=0).tolist()
    max_bounds = np.max(raw_continuous_np, axis=0).tolist()
    bounds = (min_bounds, max_bounds)

    # 🌟 บรรทัดที่หายไปคือบรรทัดนี้ครับ! ต้องประกาศ dp_kmeans ก่อน
    dp_kmeans = DPKMeans(epsilon=epsilon, bounds=bounds, n_clusters=num_bins, random_state=42)
    
    # ถึงจะสั่ง .fit() ได้
    dp_kmeans.fit(raw_continuous_np)
    
    # 2. ดึงค่าออกมาตรงๆ (ใช้ labels_ ของ Custom Class ตามที่เราแก้กันไป)
    centers = dp_kmeans.cluster_centers_
    labels = dp_kmeans.labels_
    
    # 3. Vector Quantization: แทนที่ค่าเดิมด้วยพิกัด Centroid N มิติ
    vq_data = centers[labels]
    
    # 4. Label Encoding แยกทีละคอลัมน์
    num_cols = vq_data.shape[1]
    discrete_data = np.zeros_like(vq_data, dtype=int)
    encoders = []
    json_domain = {}
    
    for col in range(num_cols):
        le = LabelEncoder()
        # แปลงทศนิยมที่ซ้ำๆ กันให้เป็น 0, 1, 2...
        discrete_data[:, col] = le.fit_transform(vq_data[:, col])
        encoders.append(le)
        # บันทึกขนาด Domain
        json_domain[col] = {'domain': len(le.classes_)}
        
    return discrete_data, encoders, json_domain

# ==========================================
# 1.5 🔄 ฟังก์ชันแปลงกลับเป็นทศนิยม (N คอลัมน์)
# ==========================================
def inverse_transform_nd(syn_discrete_features, encoders):
    num_cols = len(encoders)
    syn_continuous = np.zeros((syn_discrete_features.shape[0], num_cols), dtype=float)
    
    for col in range(num_cols):
        # ป้องกันค่า Error ที่ PrivMRF สุ่มเกิน
        safe_idx = np.clip(syn_discrete_features[:, col], 0, len(encoders[col].classes_) - 1)
        syn_continuous[:, col] = encoders[col].inverse_transform(safe_idx)
        
    return syn_continuous

# ==========================================
# 2. ไปป์ไลน์หลัก: VQ + PrivMRF Baseline
# ==========================================
def run_dp_kmeans_binning_pipeline():
    DATA_NAME = 'bean_data'
    CONTINUOUS_COLS = list(range(16)) # ไดนามิก: มีกี่คอลัมน์ก็ใส่ไป
    LABEL_COL = [-1] 
    
    # 🌟 กำหนดจำนวน Cluster ที่ต้องการทดสอบ (ปรับช่วงให้เหมาะสมกับข้อมูล)
    BIN_LIST = [200]
    EPSILON_LIST = [0.1, 0.2, 0.4, 0.8, 1.6, 3.2]
    
    TRAIN_SAMPLE_LIMIT = 10000 

    RAW_FILE_PATH = f'./data/{DATA_NAME}.csv'

    print("🚀 เริ่มต้นระบบ: VQ Multivariate DP K-Means + PrivMRF")
    print("-" * 85)

    if not os.path.exists(RAW_FILE_PATH):
        print(f"❌ ไม่พบไฟล์ {RAW_FILE_PATH}")
        return
        
    # --- โหลดและเตรียมข้อมูลจริง ---
    with open(RAW_FILE_PATH, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        headings = next(reader)
        raw_data = [row for row in reader if len(row) > 0]

    cont_headings = [headings[i] for i in CONTINUOUS_COLS]
    label_heading = [headings[LABEL_COL[0]]]
    all_headings = cont_headings + label_heading
    num_cont_cols = len(CONTINUOUS_COLS)

    cont_data = [[float(row[i]) for i in CONTINUOUS_COLS] for row in raw_data] 
    raw_continuous_np = np.array(cont_data)
    raw_label_np = np.array([[row[LABEL_COL[0]]] for row in raw_data]) 
    
    le_master = LabelEncoder()
    encoded_label_master = le_master.fit_transform(raw_label_np.ravel()).reshape(-1, 1)

    raw_scaled = np.concatenate([raw_continuous_np, encoded_label_master], axis=1)

    # ==========================================
    # 🌟 รัน Absolute TRTR (Baseline ยืนพื้น)
    # ==========================================
    print(f"\n🔥 กำลังประเมิน The Absolute TRTR Baseline (Continuous Space)...")
    np.random.seed(42)
    shuffled_idx_real = np.random.permutation(len(raw_scaled))
    shuffled_raw_scaled = raw_scaled[shuffled_idx_real]
    
    fold_size_real = len(shuffled_raw_scaled) // 5
    trtr_fold_acc = []
    trtr_fold_f1 = []
    TARGET_LABEL_INDEX = -1

    for j in range(5):
        start_r, end_r = j * fold_size_real, (j + 1) * fold_size_real
        if j == 4: end_r = len(shuffled_raw_scaled)
        
        X_real_test = shuffled_raw_scaled[start_r:end_r]
        X_real_train_full = np.concatenate([shuffled_raw_scaled[:start_r], shuffled_raw_scaled[end_r:]], axis=0)
        
        if len(X_real_train_full) > TRAIN_SAMPLE_LIMIT:
            np.random.seed(42 + j)
            sample_idx = np.random.choice(len(X_real_train_full), TRAIN_SAMPLE_LIMIT, replace=False)
            X_real_train = X_real_train_full[sample_idx]
        else:
            X_real_train = X_real_train_full

        y_train_trtr = X_real_train[:, TARGET_LABEL_INDEX].astype(int)
        X_train_trtr = np.delete(X_real_train, TARGET_LABEL_INDEX, axis=1)
        
        y_test_trtr = X_real_test[:, TARGET_LABEL_INDEX].astype(int)
        X_test_trtr = np.delete(X_real_test, TARGET_LABEL_INDEX, axis=1)

        clf_trtr = make_pipeline(StandardScaler(), svm.SVC(gamma='auto'))
        clf_trtr.fit(X_train_trtr, y_train_trtr)
        y_pred_trtr = clf_trtr.predict(X_test_trtr) 
        
        trtr_fold_acc.append(accuracy_score(y_test_trtr, y_pred_trtr))
        trtr_fold_f1.append(f1_score(y_test_trtr, y_pred_trtr, average='macro'))

    absolute_trtr_acc = np.mean(trtr_fold_acc)
    absolute_trtr_f1 = np.mean(trtr_fold_f1)
    print(f"✅ Absolute TRTR ยืนพื้นสำเร็จ! | Acc: {absolute_trtr_acc:.4f} | F1: {absolute_trtr_f1:.4f}")
    print("-" * 85)

    if not os.path.exists('./preprocess'): os.mkdir('./preprocess')
    if not os.path.exists('./out'): os.mkdir('./out')
    if not os.path.exists('./result'): os.mkdir('./result')

    summary_results = []

    # ==========================================
    # 🌟 เริ่มลูปทดสอบ VQ + PrivMRF
    # ==========================================
    for NUM_BINS in BIN_LIST:
        print(f"\n" + "🔥"*40)
        print(f"🎯 [ทดสอบ K-Means (K) = {NUM_BINS}]")
        print("🔥"*40)

        for TOTAL_EPS in EPSILON_LIST:
            print(f"\n📦 [Epsilon = {TOTAL_EPS}] กำลังทำงาน...")

            # --- 1. เตรียมข้อมูล Discrete N มิติ ---
            discrete_nd, encoders, json_domain = prepare_nd_discrete_data(
                raw_continuous_np, 
                epsilon=TOTAL_EPS, 
                num_bins=NUM_BINS
            )

            # --- 2. เอา Label ต่อท้าย ---
            processed_data = np.concatenate([discrete_nd, encoded_label_master], axis=1)
            json_domain[num_cont_cols] = {'domain': len(le_master.classes_)}

            # บันทึกไฟล์ CSV (ใช้หัวคอลัมน์เต็ม) และ JSON
            with open(f'./preprocess/{DATA_NAME}.csv', 'w', newline='', encoding='utf-8') as f:
                csv.writer(f).writerow(all_headings)
                csv.writer(f).writerows(processed_data)
            with open(f'./preprocess/{DATA_NAME}.json', 'w', encoding='utf-8') as f:
                json.dump(json_domain, f)

            # กำหนด Domain Object ตามจำนวนคอลัมน์จริงทั้งหมด
            domain_obj = Domain(json_domain, list(range(num_cont_cols + 1)))
            dummy_hierarchy = ah.get_one_level_hierarchy(domain_obj)
            ah.write_hierarchy(dummy_hierarchy, f'./preprocess/{DATA_NAME}_hierarchy.json')

            exp_name = f"VQKMeans_{NUM_BINS}_Eps_{TOTAL_EPS}"
            
            # --- 3. สร้างข้อมูลสังเคราะห์ด้วย PrivMRF ---
            syn_discrete = run_syn(DATA_NAME, exp_name, epsilon=TOTAL_EPS, task='TVD')
            syn_discrete = np.array(syn_discrete, dtype=int)
            
            # --- 4. แปลงกลับ (Inverse Transform) คืนชีพเวกเตอร์ N มิติ ---
            syn_features = syn_discrete[:, :num_cont_cols]
            syn_labels = syn_discrete[:, num_cont_cols].reshape(-1, 1)
            
            syn_continuous_features = inverse_transform_nd(syn_features, encoders)
            syn_continuous = np.concatenate([syn_continuous_features, syn_labels], axis=1)

            # --- 5. ประเมิน TRUE TSTR ---
            np.random.seed(42)
            
            shuffled_idx_real = np.random.permutation(len(raw_scaled))
            shuffled_real_raw = raw_scaled[shuffled_idx_real] 
            
            shuffled_idx_syn = np.random.permutation(len(syn_continuous))
            shuffled_syn_continuous = syn_continuous[shuffled_idx_syn]
            
            fold_size_real = len(shuffled_real_raw) // 5
            fold_size_syn = len(shuffled_syn_continuous) // 5
            
            tstr_fold_acc = []
            tstr_fold_f1 = []
            
            for j in range(5): 
                start_r, end_r = j * fold_size_real, (j + 1) * fold_size_real
                if j == 4: end_r = len(shuffled_real_raw)
                X_real_test_fold = shuffled_real_raw[start_r:end_r]
                
                start_s, end_s = j * fold_size_syn, (j + 1) * fold_size_syn
                X_syn_train_fold_full = np.concatenate([shuffled_syn_continuous[:start_s], shuffled_syn_continuous[end_s:]], axis=0)
                
                if len(X_syn_train_fold_full) > TRAIN_SAMPLE_LIMIT:
                    np.random.seed(42 + j)
                    sample_indices = np.random.choice(len(X_syn_train_fold_full), TRAIN_SAMPLE_LIMIT, replace=False)
                    X_syn_train_fold = X_syn_train_fold_full[sample_indices]
                else:
                    X_syn_train_fold = X_syn_train_fold_full

                y_train = X_syn_train_fold[:, TARGET_LABEL_INDEX].astype(int)
                X_train = np.delete(X_syn_train_fold, TARGET_LABEL_INDEX, axis=1)
                
                y_test = X_real_test_fold[:, TARGET_LABEL_INDEX].astype(int)
                X_test = np.delete(X_real_test_fold, TARGET_LABEL_INDEX, axis=1)
                
                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    continue
                    
                clf_tstr = make_pipeline(StandardScaler(), svm.SVC(gamma='auto'))
                clf_tstr.fit(X_train, y_train)
                y_pred = clf_tstr.predict(X_test)
                
                tstr_fold_acc.append(accuracy_score(y_test, y_pred))
                tstr_fold_f1.append(f1_score(y_test, y_pred, average='macro'))

            tstr_acc_final = np.mean(tstr_fold_acc) if tstr_fold_acc else 0.0
            tstr_f1_final = np.mean(tstr_fold_f1) if tstr_fold_f1 else 0.0

            print(f"   -> [K={NUM_BINS:<2} | Eps={TOTAL_EPS:<3}] TRUE TSTR Acc: {tstr_acc_final:.4f} | TRUE TSTR F1: {tstr_f1_final:.4f}")

            summary_results.append({
                'Num Bins (K)': NUM_BINS,
                'Epsilon': TOTAL_EPS,
                'Downstream Acc (TSTR)': tstr_acc_final,
                'Downstream Macro F1 (TSTR)': tstr_f1_final,
                'Absolute TRTR Acc': absolute_trtr_acc,
                'Absolute TRTR F1': absolute_trtr_f1,
                'Utility Score (Acc)': (tstr_acc_final / absolute_trtr_acc) if absolute_trtr_acc > 0 else 0.0,
                'Utility Score (F1)': (tstr_f1_final / absolute_trtr_f1) if absolute_trtr_f1 > 0 else 0.0
            })

    # ==========================================
    # 🌟 สร้างตารางสรุปจบ
    # ==========================================
    print("\n" + "🔥"*45)
    print("🎯 สรุปผล: VQ Multivariate DP K-Means + PrivMRF")
    print("🔥"*45)
    
    summary_df = pd.DataFrame(summary_results)
    summary_df.sort_values(by=['Num Bins (K)', 'Epsilon'], inplace=True)
    
    for col in ['Downstream Acc (TSTR)', 'Downstream Macro F1 (TSTR)', 'Absolute TRTR Acc', 'Absolute TRTR F1', 'Utility Score (Acc)', 'Utility Score (F1)']:
        summary_df[col] = summary_df[col].apply(lambda x: f"{x:.5f}" if isinstance(x, float) else x)
    
    print(summary_df.to_string(index=False))
    print("-" * 85)
    
    summary_df.to_csv('./result/vq_multivariate_dp_kmeans_results.csv', index=False)
    print("✅ รายงานผลถูกเซฟลง ./result/vq_multivariate_dp_kmeans_results.csv เรียบร้อยแล้วครับ!")

if __name__ == '__main__':
    run_dp_kmeans_binning_pipeline()

In [1]:
import collections
import collections.abc

# แพตช์ (Patch) เพื่อให้โค้ดเก่า (PrivMRF) ทำงานได้บน Python 3.10+
collections.Iterable = collections.abc.Iterable
collections.Mapping = collections.abc.Mapping
collections.MutableSet = collections.abc.MutableSet
collections.MutableMapping = collections.abc.MutableMapping

import matplotlib
matplotlib.use('Agg') 

import warnings
warnings.filterwarnings('ignore')

import os
import csv
import json
import random 
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn import svm

# อย่าลืม !pip install diffprivlib ก่อนรัน
# from diffprivlib.models import KMeans as DPKMeans
from custom_dp_kmeans import KMeans as DPKMeans

# ดึงเครื่องมือของ PrivMRF มาใช้งาน
from exp.evaluate import run_syn 
from PrivMRF.domain import Domain
import PrivMRF.attribute_hierarchy as ah

# ==========================================
# 1. 🎯 Core Algorithm: DP K-Means 1D Binning
# ==========================================
def dp_kmeans_1d_binning(raw_col_data, epsilon=1.0, n_clusters=70, public_bounds=None):
    X = raw_col_data.reshape(-1, 1)

    # 1. การกำหนดขอบเขต (Bounds) [กฎเหล็กของ DP]
    if public_bounds is None:
        public_bounds = (np.min(raw_col_data), np.max(raw_col_data))

    domain_min, domain_max = public_bounds

    # 🌟 [แก้ไข 1]: Split Epsilon ภายในฟังก์ชัน (ครึ่งนึงหา Center, ครึ่งนึงหา Counts)
    eps_kmeans = epsilon / 2.0
    eps_counts = epsilon / 2.0

    # 2. รัน DP K-Means
    dp_kmeans = DPKMeans(
        epsilon=eps_kmeans, # ใช้ Epsilon ที่ถูกหารแล้ว
        bounds=public_bounds, 
        n_clusters=n_clusters,
        random_state=42
    )
    dp_kmeans.fit(X)

    # 3. ดึง Center และเรียงลำดับจากซ้ายไปขวา
    original_centers = dp_kmeans.cluster_centers_.flatten()
    sorted_idx = np.argsort(original_centers)
    sorted_centers = original_centers[sorted_idx]

    # 4. การนับจำนวนข้อมูล (จำลองฉีด Noise เพื่อให้ปลอดภัย 100%)
    labels = dp_kmeans.predict(X)
    dp_counts = []
    noise_scale = 1.0 / eps_counts # ใช้ Epsilon ส่วนที่เหลือ

    for idx in sorted_idx:
        true_count = np.sum(labels == idx)
        noisy_count = true_count + np.random.laplace(0, noise_scale)
        # 🌟 [แก้ไข 2]: เอา Clip Zero (max 0) ออก ปล่อยให้ติดลบได้
        dp_counts.append(int(noisy_count))

    # 5. ตีเส้นแบ่งเขตแดน (Midpoints ระหว่าง Centers)
    edges = [domain_min - 1e-5]
    for i in range(len(sorted_centers) - 1):
        midpoint = (sorted_centers[i] + sorted_centers[i+1]) / 2.0
        edges.append(midpoint)
    edges.append(domain_max + 1e-5)

    return sorted_centers, np.array(edges), dp_counts

# ==========================================
# 1.5 🔄 ฟังก์ชัน Inverse Transform (ใช้ค่า Center ของ K-Means)
# ==========================================
def inverse_transform_kmeans(discrete_data, bin_info, num_continuous_cols):
    continuous_data = np.zeros(discrete_data.shape, dtype=float)
    
    for col in range(num_continuous_cols):
        centers = bin_info[col]['centers'] 
        col_indices = discrete_data[:, col].astype(int)
        
        safe_indices = np.clip(col_indices, 0, len(centers) - 1)
        continuous_data[:, col] = centers[safe_indices]
        
    continuous_data[:, -1] = discrete_data[:, -1]
    return continuous_data

# ==========================================
# 2. ไปป์ไลน์หลัก: DP K-Means Binning Baseline
# ==========================================
def run_dp_kmeans_binning_pipeline():
    DATA_NAME = 'bean_data'
    CONTINUOUS_COLS = list(range(16))
    LABEL_COL = [-1]
    
    # BIN_LIST = [2,3,4,5,6,7,8,9]
    BIN_LIST = [2,3,4,5,6]
    EPSILON_LIST = [0.1, 0.2, 0.4, 0.8, 1.6, 3.2]
    
    TRAIN_SAMPLE_LIMIT = 10000 

    RAW_FILE_PATH = f'./data/{DATA_NAME}.csv'

    print("🚀 เริ่มต้นระบบ: DP K-Means Binning Baseline")
    print("-" * 85)

    if not os.path.exists(RAW_FILE_PATH):
        print(f"❌ ไม่พบไฟล์ {RAW_FILE_PATH}")
        return
        
    with open(RAW_FILE_PATH, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        headings = next(reader)
        raw_data = [row for row in reader if len(row) > 0]

    cont_headings = [headings[i] for i in CONTINUOUS_COLS]
    label_heading = [headings[LABEL_COL[0]]]
    all_headings = cont_headings + label_heading

    cont_data = [[float(row[i]) for i in CONTINUOUS_COLS] for row in raw_data] 
    raw_continuous_np = np.array(cont_data)
    raw_label_np = np.array([[row[LABEL_COL[0]]] for row in raw_data]) 
    
    le_master = LabelEncoder()
    encoded_label_master = le_master.fit_transform(raw_label_np.ravel()).reshape(-1, 1)

    raw_scaled = np.concatenate([raw_continuous_np, encoded_label_master], axis=1)

    print(f"\n🔥 กำลังประเมิน The Absolute TRTR Baseline (Continuous Space)...")
    np.random.seed(42)
    shuffled_idx_real = np.random.permutation(len(raw_scaled))
    shuffled_raw_scaled = raw_scaled[shuffled_idx_real]
    
    fold_size_real = len(shuffled_raw_scaled) // 5
    trtr_fold_acc = []
    trtr_fold_f1 = []
    TARGET_LABEL_INDEX = -1

    for j in range(5):
        start_r, end_r = j * fold_size_real, (j + 1) * fold_size_real
        if j == 4: end_r = len(shuffled_raw_scaled)
        
        X_real_test = shuffled_raw_scaled[start_r:end_r]
        X_real_train_full = np.concatenate([shuffled_raw_scaled[:start_r], shuffled_raw_scaled[end_r:]], axis=0)
        
        if len(X_real_train_full) > TRAIN_SAMPLE_LIMIT:
            np.random.seed(42 + j)
            sample_idx = np.random.choice(len(X_real_train_full), TRAIN_SAMPLE_LIMIT, replace=False)
            X_real_train = X_real_train_full[sample_idx]
        else:
            X_real_train = X_real_train_full

        y_train_trtr = X_real_train[:, TARGET_LABEL_INDEX].astype(int)
        X_train_trtr = np.delete(X_real_train, TARGET_LABEL_INDEX, axis=1)
        
        y_test_trtr = X_real_test[:, TARGET_LABEL_INDEX].astype(int)
        X_test_trtr = np.delete(X_real_test, TARGET_LABEL_INDEX, axis=1)

        clf_trtr = make_pipeline(StandardScaler(), svm.SVC(gamma='auto'))
        clf_trtr.fit(X_train_trtr, y_train_trtr)
        y_pred_trtr = clf_trtr.predict(X_test_trtr) 
        
        trtr_fold_acc.append(accuracy_score(y_test_trtr, y_pred_trtr))
        trtr_fold_f1.append(f1_score(y_test_trtr, y_pred_trtr, average='macro'))

    absolute_trtr_acc = np.mean(trtr_fold_acc)
    absolute_trtr_f1 = np.mean(trtr_fold_f1)
    print(f"✅ Absolute TRTR ยืนพื้นสำเร็จ! | Acc: {absolute_trtr_acc:.4f} | F1: {absolute_trtr_f1:.4f}")
    print("-" * 85)

    if not os.path.exists('./preprocess'): os.mkdir('./preprocess')
    if not os.path.exists('./out'): os.mkdir('./out')
    if not os.path.exists('./result'): os.mkdir('./result')

    summary_results = []

    for NUM_BINS in BIN_LIST:
        print(f"\n" + "🔥"*40)
        print(f"🎯 [ทดสอบ DP K-Means (K) = {NUM_BINS}]")
        print("🔥"*40)

        for TOTAL_EPS in EPSILON_LIST:
            print(f"\n📦 [Epsilon = {TOTAL_EPS}] กำลังทำงาน...")

            # 🌟 [แก้ไข 3]: Split Epsilon ระดับภาพรวม เพื่อความปลอดภัยทางทฤษฎี DP
            # แบ่ง 50% ให้กระบวนการ K-Means Binning และ 50% ให้ PrivMRF สังเคราะห์ข้อมูล
            eps_binning_total = TOTAL_EPS / 2.0
            eps_privmrf = TOTAL_EPS / 2.0
            
            # ต้องหาร Epsilon ต่อคอลัมน์ด้วย ไม่อย่างนั้นงบจะเกินตามจำนวนคอลัมน์
            eps_per_col = eps_binning_total / len(CONTINUOUS_COLS)

            json_domain = {}
            processed_data = np.zeros((len(cont_data), len(all_headings)), dtype=int)
            bin_info = {}

            # --- 1. ทำ DP K-Means Binning ทีละคอลัมน์ ---
            for col in range(len(CONTINUOUS_COLS)):
                col_values = raw_continuous_np[:, col]
                
                # โยนงบที่หารมาแล้ว (eps_per_col) เข้าไปทำงานทีละคอลัมน์
                centers, edges, _ = dp_kmeans_1d_binning(
                    col_values, 
                    epsilon=eps_per_col, 
                    n_clusters=NUM_BINS
                )
                
                bin_info[col] = {'centers': centers, 'edges': edges, 'num_bins': NUM_BINS}
                json_domain[col] = {'domain': NUM_BINS}
                
                b_idx = np.searchsorted(edges, col_values, side='right') - 1
                b_idx = np.clip(b_idx, 0, NUM_BINS - 1)
                
                processed_data[:, col] = b_idx

            processed_data[:, -1] = encoded_label_master.ravel()
            json_domain[len(CONTINUOUS_COLS)] = {'domain': len(le_master.classes_)}

            with open(f'./preprocess/{DATA_NAME}.csv', 'w', newline='', encoding='utf-8') as f:
                csv.writer(f).writerow(all_headings)
                csv.writer(f).writerows(processed_data)
            with open(f'./preprocess/{DATA_NAME}.json', 'w', encoding='utf-8') as f:
                json.dump(json_domain, f)

            domain_obj = Domain(json_domain, list(range(len(all_headings))))
            dummy_hierarchy = ah.get_one_level_hierarchy(domain_obj)
            ah.write_hierarchy(dummy_hierarchy, f'./preprocess/{DATA_NAME}_hierarchy.json')

            exp_name = f"DPKMeans_{NUM_BINS}_Eps_{TOTAL_EPS}"
            
            # --- 2. สร้างข้อมูลสังเคราะห์ (PrivMRF) ---
            # ใช้ Epsilon อีกครึ่งหนึ่งที่สำรองไว้
            syn_discrete = run_syn(DATA_NAME, exp_name, epsilon=eps_privmrf, task='TVD')
            syn_discrete = np.array(syn_discrete, dtype=int)
            
            num_cont_cols = len(CONTINUOUS_COLS)
            
            # 🌟 แปลงกลับด้วย Center ของ DP K-Means (Inverse Transform)
            syn_continuous = inverse_transform_kmeans(syn_discrete, bin_info, num_cont_cols)

            # --- 3. ประเมิน TRUE TSTR ---
            np.random.seed(42)
            
            shuffled_idx_real = np.random.permutation(len(raw_scaled))
            shuffled_real_raw = raw_scaled[shuffled_idx_real] 
            
            shuffled_idx_syn = np.random.permutation(len(syn_continuous))
            shuffled_syn_continuous = syn_continuous[shuffled_idx_syn]
            
            fold_size_real = len(shuffled_real_raw) // 5
            fold_size_syn = len(shuffled_syn_continuous) // 5
            
            tstr_fold_acc = []
            tstr_fold_f1 = []
            
            for j in range(5): 
                start_r, end_r = j * fold_size_real, (j + 1) * fold_size_real
                if j == 4: end_r = len(shuffled_real_raw)
                X_real_test_fold = shuffled_real_raw[start_r:end_r]
                
                start_s, end_s = j * fold_size_syn, (j + 1) * fold_size_syn
                X_syn_train_fold_full = np.concatenate([shuffled_syn_continuous[:start_s], shuffled_syn_continuous[end_s:]], axis=0)
                
                if len(X_syn_train_fold_full) > TRAIN_SAMPLE_LIMIT:
                    np.random.seed(42 + j)
                    sample_indices = np.random.choice(len(X_syn_train_fold_full), TRAIN_SAMPLE_LIMIT, replace=False)
                    X_syn_train_fold = X_syn_train_fold_full[sample_indices]
                else:
                    X_syn_train_fold = X_syn_train_fold_full

                y_train = X_syn_train_fold[:, TARGET_LABEL_INDEX].astype(int)
                X_train = np.delete(X_syn_train_fold, TARGET_LABEL_INDEX, axis=1)
                
                y_test = X_real_test_fold[:, TARGET_LABEL_INDEX].astype(int)
                X_test = np.delete(X_real_test_fold, TARGET_LABEL_INDEX, axis=1)
                
                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    continue
                    
                clf_tstr = make_pipeline(StandardScaler(), svm.SVC(gamma='auto'))
                clf_tstr.fit(X_train, y_train)
                y_pred = clf_tstr.predict(X_test)
                
                tstr_fold_acc.append(accuracy_score(y_test, y_pred))
                tstr_fold_f1.append(f1_score(y_test, y_pred, average='macro'))

            tstr_acc_final = np.mean(tstr_fold_acc) if tstr_fold_acc else 0.0
            tstr_f1_final = np.mean(tstr_fold_f1) if tstr_fold_f1 else 0.0

            print(f"   -> [K={NUM_BINS:<2} | Eps={TOTAL_EPS:<3}] TRUE TSTR Acc: {tstr_acc_final:.4f} | TRUE TSTR F1: {tstr_f1_final:.4f}")

            summary_results.append({
                'Num Bins (K)': NUM_BINS,
                'Epsilon': TOTAL_EPS,
                'Downstream Acc (TSTR)': tstr_acc_final,
                'Downstream Macro F1 (TSTR)': tstr_f1_final,
                'Absolute TRTR Acc': absolute_trtr_acc,
                'Absolute TRTR F1': absolute_trtr_f1,
                'Utility Score (Acc)': (tstr_acc_final / absolute_trtr_acc) if absolute_trtr_acc > 0 else 0.0,
                'Utility Score (F1)': (tstr_f1_final / absolute_trtr_f1) if absolute_trtr_f1 > 0 else 0.0
            })

    # ==========================================
    # 🌟 สร้างตารางสรุปจบ
    # ==========================================
    print("\n" + "🔥"*45)
    print("🎯 สรุปผล: DP K-Means Binning (True TSTR vs Absolute TRTR)")
    print("🔥"*45)
    
    summary_df = pd.DataFrame(summary_results)
    summary_df.sort_values(by=['Num Bins (K)', 'Epsilon'], inplace=True)
    
    for col in ['Downstream Acc (TSTR)', 'Downstream Macro F1 (TSTR)', 'Absolute TRTR Acc', 'Absolute TRTR F1', 'Utility Score (Acc)', 'Utility Score (F1)']:
        summary_df[col] = summary_df[col].apply(lambda x: f"{x:.5f}" if isinstance(x, float) else x)
    
    print(summary_df.to_string(index=False))
    print("-" * 85)
    
    summary_df.to_csv('./result/dp_kmeans_binning_true_tstr_results.csv', index=False)
    print("✅ รายงานผลถูกเซฟลง ./result/dp_kmeans_binning_true_tstr_results.csv เรียบร้อยแล้วครับ!")

if __name__ == '__main__':
    run_dp_kmeans_binning_pipeline()

🚀 เริ่มต้นระบบ: DP K-Means Binning Baseline
-------------------------------------------------------------------------------------

🔥 กำลังประเมิน The Absolute TRTR Baseline (Continuous Space)...
✅ Absolute TRTR ยืนพื้นสำเร็จ! | Acc: 0.9299 | F1: 0.9412
-------------------------------------------------------------------------------------

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🎯 [ทดสอบ DP K-Means (K) = 2]
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

📦 [Epsilon = 0.1] กำลังทำงาน...
    read csv data ./preprocess/bean_data.csv
PrivMRF
theta: 6
privacy budget:            2.9962975531816483e-05
estimated noise scale:     357.29196524411975
max 2way measure dom:      6.357545707606376
max high way measure dom:  6.357545707606376
max edge number:           136.0
TVD_noise:                 0.3126388816397756
data num:                  13629
construct attribute graph
data entropy: 4.682198414569305
  init score: -9.7685 query_num: 17 size: 3.9000e+01, edge_num: 0 entropy: 9.77
average TVD and no

KeyboardInterrupt: 